In [ ]:
# Install required libraries
!pip install -q transformers datasets peft wandb accelerate
!pip install -q torch torchvision torchaudio

print("✅ Installation complete!")

In [ ]:
import wandb

# Log into W&B
wandb.login()

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import load_dataset
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftModel
)
import numpy as np
import wandb

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Load WikiText-103 (same as Week 3)
dataset = load_dataset("wikitext", "wikitext-103-v1")

print(f"Train samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")
print(f"Test samples: {len(dataset['test'])}")

# Take a smaller subset for faster training during experimentation
# You can increase this later for final runs
train_dataset = dataset['train'].shuffle(seed=42).select(range(10000))
val_dataset = dataset['validation'].shuffle(seed=42).select(range(1000))

print(f"\n Using {len(train_dataset)} training samples")
print(f" Using {len(val_dataset)} validation samples")

In [ ]:
# Load tokenizer
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Tokenization function
def tokenize_function(examples):
    # Remove empty lines
    texts = [text for text in examples['text'] if len(text) > 0 and not text.isspace()]

    # Tokenize
    return tokenizer(
        texts,
        truncation=True,
        max_length=512,
        padding=False,
        return_overflowing_tokens=False
    )

# Tokenize datasets
print("Tokenizing datasets...")
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing train"
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=val_dataset.column_names,
    desc="Tokenizing validation"
)

print(" Tokenization complete!")
print(f"Tokenized train size: {len(tokenized_train)}")
print(f"Tokenized val size: {len(tokenized_val)}")

In [ ]:
# Load base model
print("Loading GPT-2 model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

# LoRA Configuration
# Starting with rank=8, alpha=16 (good default)
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,  # Causal language modeling
    r=8,                            # LoRA rank - LOW RANK!
    lora_alpha=16,                  # LoRA alpha scaling
    lora_dropout=0.1,               # Dropout for LoRA layers
    target_modules=["c_attn"],      # Which layers to apply LoRA to
    bias="none"
)

# Apply LoRA to the model
model = get_peft_model(base_model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

In [ ]:
# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # We're doing causal LM, not masked LM
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./lora_gpt2_r8_alpha16",

    # Training hyperparameters
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-5,

    # Optimization
    warmup_steps=100,
    weight_decay=0.01,

    # Logging and saving
    logging_steps=50,
    eval_strategy="steps",  # Changed from evaluation_strategy
    eval_steps=200,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    # W&B integration
    report_to="wandb",
    run_name="lora_r8_alpha16_lr3e-5",

    # Performance
    fp16=True,  # Mixed precision training
    gradient_accumulation_steps=2,
)

print(" Training arguments configured!")
print(f"Total training steps: ~{len(tokenized_train) // 8 // 2 * 3}")
print(f"Evaluations every 200 steps")

In [ ]:
# Initialize W&B run
wandb.init(
    project="llm-capstone-week4",
    name="lora_r8_alpha16_lr3e-5",
    config={
        "lora_r": 8,
        "lora_alpha": 16,
        "learning_rate": 3e-5,
        "batch_size": 8,
        "epochs": 3,
        "model": "gpt2",
        "dataset": "wikitext-103",
        "train_samples": len(tokenized_train),
        "val_samples": len(tokenized_val)
    }
)

print(" W&B run initialized!")
print(f" Track your run at: {wandb.run.get_url()}")

In [ ]:
# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

print("Starting training...")

# Train!
trainer.train()

print("\n Training complete!")

In [ ]:
# Final evaluation
print("Running final evaluation...")
eval_results = trainer.evaluate()

# Calculate perplexity
perplexity = np.exp(eval_results['eval_loss'])

print("\n" + "="*50)
print("FINAL RESULTS - LoRA r=8, alpha=16, lr=3e-5")
print("="*50)
print(f"Validation Loss: {eval_results['eval_loss']:.4f}")
print(f"Perplexity: {perplexity:.2f}")
print("="*50)

# Log to W&B
wandb.log({
    "final_perplexity": perplexity,
    "final_loss": eval_results['eval_loss']
})

# Save the LoRA adapter
print("\nSaving LoRA model...")
model.save_pretrained("./lora_gpt2_r8_alpha16_final")
tokenizer.save_pretrained("./lora_gpt2_r8_alpha16_final")

print("Model saved to: ./lora_gpt2_r8_alpha16_final")

# Finish W&B run
wandb.finish()

print("\n First experiment complete!")

In [ ]:
# EXPERIMENT 2: Lower rank (r=4)
print("="*60)
print(" EXPERIMENT 2: LoRA r=4, alpha=16, lr=3e-5")
print("="*60)

# Load fresh base model
base_model_2 = AutoModelForCausalLM.from_pretrained("gpt2", device_map="auto")

# LoRA config with r=4
lora_config_2 = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=4,                    # Lower rank
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["c_attn"],
    bias="none"
)

model_2 = get_peft_model(base_model_2, lora_config_2)
model_2.print_trainable_parameters()

# Training arguments
training_args_2 = TrainingArguments(
    output_dir="./lora_gpt2_r4_alpha16",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-5,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    report_to="wandb",
    run_name="lora_r4_alpha16_lr3e-5",
    fp16=True,
    gradient_accumulation_steps=2,
)

# Initialize W&B
wandb.init(
    project="llm-capstone-week4",
    name="lora_r4_alpha16_lr3e-5",
    config={
        "lora_r": 4,
        "lora_alpha": 16,
        "learning_rate": 3e-5,
        "batch_size": 8,
        "epochs": 3,
    }
)

# Create trainer
trainer_2 = Trainer(
    model=model_2,
    args=training_args_2,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

print("\n Starting Experiment 2...")
trainer_2.train()

# Save results
final_loss_2 = trainer_2.state.log_history[-1]['eval_loss']
perplexity_2 = np.exp(final_loss_2)

print(f"\n Experiment 2 Results:")
print(f"Loss: {final_loss_2:.4f}, Perplexity: {perplexity_2:.2f}")

model_2.save_pretrained("./lora_gpt2_r4_alpha16_final")
wandb.finish()

print(" Experiment 2 complete!\n")

In [ ]:
# Extract final results from Experiment 2

final_loss_2 = 3.570385  # From the last eval step
perplexity_2 = np.exp(final_loss_2)

print(f"\n Experiment 2 Results:")
print(f"Loss: {final_loss_2:.4f}, Perplexity: {perplexity_2:.2f}")

# Compare to Experiment 1
print(f"\n COMPARISON:")
print(f"Experiment 1 (r=8): Perplexity = 31.89")
print(f"Experiment 2 (r=4): Perplexity = {perplexity_2:.2f}")
print(f"Difference: {perplexity_2 - 31.89:.2f}")

# Also note trainable parameters
print(f"\nTrainable parameters:")
print(f"r=8: 294,912 params")
print(f"r=4: 147,456 params (50% fewer!)")

model_2.save_pretrained("./lora_gpt2_r4_alpha16_final")
wandb.finish()

print("\n Experiment 2 complete and saved!")

In [ ]:
# EXPERIMENT 3: Higher rank (r=16)
print("="*60)
print(" EXPERIMENT 3: LoRA r=16, alpha=32, lr=3e-5")
print("="*60)

# Load fresh base model
base_model_3 = AutoModelForCausalLM.from_pretrained("gpt2", device_map="auto")

# LoRA config with r=16
lora_config_3 = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                   # Higher rank
    lora_alpha=32,          # Scale alpha proportionally
    lora_dropout=0.1,
    target_modules=["c_attn"],
    bias="none"
)

model_3 = get_peft_model(base_model_3, lora_config_3)
model_3.print_trainable_parameters()

# Training arguments
training_args_3 = TrainingArguments(
    output_dir="./lora_gpt2_r16_alpha32",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-5,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    report_to="wandb",
    run_name="lora_r16_alpha32_lr3e-5",
    fp16=True,
    gradient_accumulation_steps=2,
)

# Initialize W&B
wandb.init(
    project="llm-capstone-week4",
    name="lora_r16_alpha32_lr3e-5",
    config={
        "lora_r": 16,
        "lora_alpha": 32,
        "learning_rate": 3e-5,
        "batch_size": 8,
        "epochs": 3,
    }
)

# Create trainer
trainer_3 = Trainer(
    model=model_3,
    args=training_args_3,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

print("\n Starting Experiment 3...")
trainer_3.train()

print("\n Experiment 3 training complete!")

In [ ]:
# Extract final results from Experiment 3
final_loss_3 = 3.504495  # From step 1200
perplexity_3 = np.exp(final_loss_3)

print(f"\n Experiment 3 Results:")
print(f"Loss: {final_loss_3:.4f}, Perplexity: {perplexity_3:.2f}")

# Compare all three
print(f"\n COMPARISON SO FAR:")
print("="*60)
print(f"Experiment 1 (r=8,  294K params): Perplexity = 31.89")
print(f"Experiment 2 (r=4,  147K params): Perplexity = 35.53")
print(f"Experiment 3 (r=16, 590K params): Perplexity = {perplexity_3:.2f}")
print("="*60)

print(f"\n Key Insights:")
print(f"r=4 → r=8:  {35.53 - 31.89:.2f} improvement (2x params)")
print(f"r=8 → r=16: {31.89 - perplexity_3:.2f} improvement (2x params)")

model_3.save_pretrained("./lora_gpt2_r16_alpha32_final")
wandb.finish()

print("\n Experiment 3 complete and saved!")

In [ ]:
# EXPERIMENT 4: Higher learning rate
print("="*60)
print(" EXPERIMENT 4: LoRA r=8, alpha=16, lr=5e-5")
print("="*60)

# Load fresh base model
base_model_4 = AutoModelForCausalLM.from_pretrained("gpt2", device_map="auto")

# LoRA config - same as Exp 1, but higher LR
lora_config_4 = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["c_attn"],
    bias="none"
)

model_4 = get_peft_model(base_model_4, lora_config_4)
model_4.print_trainable_parameters()

# Training arguments - HIGHER LEARNING RATE
training_args_4 = TrainingArguments(
    output_dir="./lora_gpt2_r8_alpha16_lr5e5",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,         # HIGHER!
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    report_to="wandb",
    run_name="lora_r8_alpha16_lr5e-5",
    fp16=True,
    gradient_accumulation_steps=2,
)

# Initialize W&B
wandb.init(
    project="llm-capstone-week4",
    name="lora_r8_alpha16_lr5e-5",
    config={
        "lora_r": 8,
        "lora_alpha": 16,
        "learning_rate": 5e-5,  # Higher LR
        "batch_size": 8,
        "epochs": 3,
    }
)

# Create trainer
trainer_4 = Trainer(
    model=model_4,
    args=training_args_4,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

print("\n Starting Experiment 4 ...")
trainer_4.train()

print("\n Experiment 4 training complete!")

In [ ]:
# Extract final results from Experiment 4
final_loss_4 = 3.483853  # From step 1200
perplexity_4 = np.exp(final_loss_4)

print(f"\n Experiment 4 Results:")
print(f"Loss: {final_loss_4:.4f}, Perplexity: {perplexity_4:.2f}")

# COMPLETE COMPARISON
print("\n" + "="*70)
print(" FINAL COMPARISON - ALL 4 EXPERIMENTS")
print("="*70)
print(f"{'Config':<25} {'Params':<12} {'Perplexity':<12} {'Rank'}")
print("-"*70)
print(f"Exp 1: r=8,  lr=3e-5     294,912      31.89         2nd")
print(f"Exp 2: r=4,  lr=3e-5     147,456      35.53        4th")
print(f"Exp 3: r=16, lr=3e-5     589,824      33.26        3rd")
print(f"Exp 4: r=8,  lr=5e-5     294,912      {perplexity_4:.2f}        {' WINNER!' if perplexity_4 < 31.89 else '🥈 2nd' if perplexity_4 < 33.26 else '3rd'}")
print("="*70)

print(f"\n KEY INSIGHTS:")
print(f"1. Higher learning rate (5e-5) {'IMPROVED' if perplexity_4 < 31.89 else 'did not improve'} over 3e-5")
print(f"2. r=8 is optimal rank (r=16 overfits, r=4 underfits)")
print(f"3. Best config: r=8, alpha=16, lr={'5e-5' if perplexity_4 < 31.89 else '3e-5'}")

model_4.save_pretrained("./lora_gpt2_r8_alpha16_lr5e5_final")
wandb.finish()

print("\n All 4 experiments complete and saved!")

In [ ]:
print("\n" + "="*70)
print("PART 2: SAMPLING STRATEGIES")
print("="*70 + "\n")

In [ ]:
# Load the best LoRA model from Experiment 1
print("Loading best LoRA model (r=8, lr=3e-5)...")

from peft import PeftModel

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Load base model
base_model = AutoModelForCausalLM.from_pretrained("gpt2", device_map="auto")

# Load LoRA adapter
model = PeftModel.from_pretrained(base_model, "./lora_gpt2_r8_alpha16_final")
model.eval()  # Set to evaluation mode

print(" Best model loaded!")
print("Ready for text generation experiments!")

In [ ]:
# Define test prompts
test_prompts = [
    "The future of artificial intelligence is",
    "In the year 2050, technology will",
    "The most important scientific discovery was",
    "Climate change requires us to",
    "The key to success in life is"
]

print(" Test prompts ready:")
for i, prompt in enumerate(test_prompts, 1):
    print(f"{i}. {prompt}")

In [ ]:
import torch

# Function to generate text with different sampling strategies
def generate_with_strategy(prompt, strategy_name, **generation_kwargs):
    """
    Generate text using specified sampling strategy
    """
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,  # Generate 50 new tokens
            pad_token_id=tokenizer.eos_token_id,
            **generation_kwargs
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text

print(" Generation function ready!")

In [ ]:
# Define all sampling strategies to test
sampling_strategies = {
    "1. Greedy (Baseline)": {
        "do_sample": False,
    },

    "2. Temperature = 0.7 (Low)": {
        "do_sample": True,
        "temperature": 0.7,
    },

    "3. Temperature = 1.0 (Medium)": {
        "do_sample": True,
        "temperature": 1.0,
    },

    "4. Temperature = 1.3 (High)": {
        "do_sample": True,
        "temperature": 1.3,
    },

    "5. Top-k = 50": {
        "do_sample": True,
        "top_k": 50,
        "temperature": 1.0,
    },

    "6. Nucleus (top-p = 0.9)": {
        "do_sample": True,
        "top_p": 0.9,
        "temperature": 1.0,
    },

    "7. Nucleus (top-p = 0.95)": {
        "do_sample": True,
        "top_p": 0.95,
        "temperature": 1.0,
    },

    "8. Combined (temp=0.8, top-p=0.95)": {
        "do_sample": True,
        "temperature": 0.8,
        "top_p": 0.95,
    }
}

print(" Testing 8 different sampling strategies!")
print("This will take ~5 minutes...\n")

# Store all results
all_results = {}

# Test each strategy with first prompt
test_prompt = test_prompts[0]  # "The future of artificial intelligence is"

print(f" Prompt: '{test_prompt}'\n")
print("="*80)

for strategy_name, params in sampling_strategies.items():
    print(f"\n{strategy_name}")
    print("-"*80)

    generated = generate_with_strategy(test_prompt, strategy_name, **params)

    # Store result
    all_results[strategy_name] = generated

    # Print only the generated part (remove prompt)
    generated_only = generated[len(test_prompt):].strip()
    print(f"{generated_only}")
    print()

print("="*80)
print(" All sampling strategies tested!")

In [ ]:
# Calculate diversity metrics for each output
import re

def calculate_diversity_metrics(text):
    """Calculate distinct-1, distinct-2 ratios"""
    # Tokenize
    words = text.lower().split()

    # Distinct-1 (unique unigrams / total unigrams)
    distinct_1 = len(set(words)) / len(words) if len(words) > 0 else 0

    # Distinct-2 (unique bigrams / total bigrams)
    bigrams = [f"{words[i]} {words[i+1]}" for i in range(len(words)-1)]
    distinct_2 = len(set(bigrams)) / len(bigrams) if len(bigrams) > 0 else 0

    # Count repetitions
    word_counts = {}
    for word in words:
        word_counts[word] = word_counts.get(word, 0) + 1

    max_repetition = max(word_counts.values()) if word_counts else 0

    return distinct_1, distinct_2, max_repetition, len(words)

# Analyze all results
print("="*80)
print(" DIVERSITY ANALYSIS")
print("="*80)
print(f"{'Strategy':<35} {'Distinct-1':<12} {'Distinct-2':<12} {'Max Rep':<10} {'Words'}")
print("-"*80)

diversity_results = {}

for strategy_name, text in all_results.items():
    # Remove prompt from analysis
    generated_only = text[len(test_prompt):].strip()
    d1, d2, max_rep, word_count = calculate_diversity_metrics(generated_only)

    diversity_results[strategy_name] = {
        'distinct_1': d1,
        'distinct_2': d2,
        'max_repetition': max_rep,
        'word_count': word_count
    }

    print(f"{strategy_name:<35} {d1:<12.3f} {d2:<12.3f} {max_rep:<10} {word_count}")

print("="*80)
print("\n Interpretation:")
print("- Distinct-1: Higher = more unique words (diversity)")
print("- Distinct-2: Higher = more unique word pairs (less repetition)")
print("- Max Rep: Lower = fewer repeated words")
print("- Greedy has MASSIVE repetition problem!")

In [ ]:
# Testing with 2 more diverse prompts
additional_prompts = [
    "Climate change requires us to",
    "The key to success in life is"
]

# Test best 3 methods only (for speed)
best_strategies = {
    "Greedy": {"do_sample": False},
    "Nucleus (0.9)": {"do_sample": True, "top_p": 0.9, "temperature": 1.0},
    "Combined": {"do_sample": True, "temperature": 0.8, "top_p": 0.95}
}

print("="*80)
print(" ADDITIONAL PROMPT TESTING")
print("="*80)

for prompt in additional_prompts:
    print(f"\n Prompt: '{prompt}'")
    print("-"*80)

    for strategy_name, params in best_strategies.items():
        generated = generate_with_strategy(prompt, strategy_name, **params)
        generated_only = generated[len(prompt):].strip()

        # Quick diversity check
        d1, d2, max_rep, wc = calculate_diversity_metrics(generated_only)

        print(f"\n{strategy_name} (D1={d1:.2f}, MaxRep={max_rep}):")
        print(f"  {generated_only[:100]}...")  # First 100 chars

    print()

print("="*80)
print(" Multi-prompt testing complete!")

In [ ]:
# Create comprehensive summary for report
print("="*80)
print("SAMPLING STRATEGIES - FINAL SUMMARY")
print("="*80)

print("\n WINNER: Nucleus Sampling (top-p = 0.9)")
print("   - Highest diversity (94.9% unique words)")
print("   - No repetition loops")
print("   - Coherent, natural text")

print("\n WORST: Greedy Decoding")
print("   - Severe repetition (only 34-58% unique words)")
print("   - Gets stuck in loops")
print("   - Unusable for production")

print("\n KEY INSIGHTS FOR REPORT:")
print("1. Greedy decoding FAILS for open-ended generation")
print("2. Nucleus sampling (top-p=0.9) optimal for quality + diversity")
print("3. Temperature alone not enough - needs top-p/top-k")
print("4. Combined (temp=0.8, top-p=0.95) good for creative tasks")

print("\n QUANTITATIVE RESULTS:")
print("   Greedy:      Distinct-1 = 0.347, Max Repetition = 5-7")
print("   Nucleus 0.9: Distinct-1 = 0.949, Max Repetition = 2-3")
print("   Improvement: 2.7x more diverse!")

print("\n Sampling strategies analysis COMPLETE!")
print("="*80)

In [ ]:
print("\n" + "="*70)
print("PART 3: ATTENTION VISUALIZATION")
print("="*70 + "\n")

# Install BertViz
!pip install -q bertviz

print(" BertViz installed!")

In [ ]:
from bertviz import model_view, head_view
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

# Using base GPT-2 for visualization (LoRA uses same architecture)
# Note: BertViz works best with the base model structure

viz_model_name = "gpt2"
viz_tokenizer = GPT2Tokenizer.from_pretrained(viz_model_name)
viz_model = GPT2LMHeadModel.from_pretrained(viz_model_name, output_attentions=True)
viz_model.eval()
viz_model.to("cuda")

print(" Visualization model loaded!")
print("Note: Using base GPT-2 to show attention patterns")

In [ ]:
def visualize_attention(text, model, tokenizer):
    """
    Visualize attention patterns for given text
    """
    # Tokenize
    inputs = tokenizer.encode(text, return_tensors='pt').to("cuda")

    # Get model outputs with attention
    with torch.no_grad():
        outputs = model(inputs, output_attentions=True)

    # Get attention weights
    attention = outputs.attentions  # Tuple of attention matrices

    # Convert tokens for display
    tokens = tokenizer.convert_ids_to_tokens(inputs[0])

    return attention, tokens, inputs

print(" Visualization function ready!")


In [ ]:
# Test sentences showing different attention patterns
test_sentences = [
    "The cat sat on the mat",
    "Artificial intelligence will transform society",
    "She went to the store because she needed milk"
]

print("="*70)
print(" GENERATING ATTENTION VISUALIZATIONS")
print("="*70)

# We'll analyze the first sentence in detail
sentence = test_sentences[0]
print(f"\n Analyzing: '{sentence}'")

attention, tokens, inputs = visualize_attention(sentence, viz_model, viz_tokenizer)

print(f"\n Attention captured!")
print(f"   - Number of layers: {len(attention)}")
print(f"   - Number of heads per layer: {attention[0].shape[1]}")
print(f"   - Tokens: {tokens}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_attention_heatmap(attention_matrix, tokens, layer_num, head_num):
    """
    Plot attention heatmap for specific layer and head
    """
    # Get attention for specific layer and head
    attn = attention_matrix[layer_num][0, head_num].cpu().numpy()

    fig, ax = plt.subplots(figsize=(10, 8))

    # Create heatmap
    im = ax.imshow(attn, cmap='viridis', aspect='auto')

    # Set ticks and labels
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right')
    ax.set_yticklabels(tokens)

    # Labels
    ax.set_xlabel('Key (attending to)', fontsize=12)
    ax.set_ylabel('Query (attending from)', fontsize=12)
    title = f'Attention Pattern - Layer {layer_num}, Head {head_num}'
    ax.set_title(title, fontsize=14, fontweight='bold')

    # Colorbar
    plt.colorbar(im, ax=ax, label='Attention Weight')

    # Add grid
    ax.set_xticks(np.arange(len(tokens))-.5, minor=True)
    ax.set_yticks(np.arange(len(tokens))-.5, minor=True)
    ax.grid(which="minor", color="w", linestyle='-', linewidth=2)

    plt.tight_layout()
    plt.show()

print(" Heatmap function ready!")

In [ ]:
print("="*70)
print(" ATTENTION HEATMAPS")
print("="*70)

# Visualize different layers and heads
print("\n 1 Early Layer (Layer 2, Head 0) - Learning Basic Patterns")
plot_attention_heatmap(attention, tokens, layer_num=2, head_num=0)

print("\n 2 Middle Layer (Layer 6, Head 5) - Complex Dependencies")
plot_attention_heatmap(attention, tokens, layer_num=6, head_num=5)

print("\n 3 Late Layer (Layer 11, Head 8) - High-Level Features")
plot_attention_heatmap(attention, tokens, layer_num=11, head_num=8)

print("\n Attention visualizations complete!")

In [ ]:
# Analyze a more complex sentence
complex_sentence = "Artificial intelligence will transform society"

print("="*70)
print(" ANALYZING COMPLEX SENTENCE")
print("="*70)
print(f"\n Sentence: '{complex_sentence}'")

attention2, tokens2, inputs2 = visualize_attention(complex_sentence, viz_model, viz_tokenizer)

print(f"\n Tokens: {tokens2}")
print(f"   Length: {len(tokens2)} tokens")

# Visualize final layer to see high-level relationships
print("\n Final Layer Attention (Layer 11, Head 3):")
plot_attention_heatmap(attention2, tokens2, layer_num=11, head_num=3)

print("\n Complex sentence analysis complete!")

In [ ]:
print("="*70)
print(" ATTENTION VISUALIZATION - KEY FINDINGS")
print("="*70)

print("\ DISCOVERED:")
print("\n1. LAYER PROGRESSION:")
print("   - Early layers (2): Basic causal masking, positional patterns")
print("   - Middle layers (6): Focus on anchor words (sentence starts)")
print("   - Late layers (11): Complex semantic relationships")

print("\n2. ATTENTION PATTERNS FOUND:")
print("   - Diagonal patterns = Causal attention (can't see future)")
print("   - Vertical stripes = All tokens attend to one important token")
print("   - Specific cells = Semantic relationships (subject-verb)")

print("\n3. SPECIFIC EXAMPLES:")
print("   - 'transform' attends to 'intelligence' (actor-action)")
print("   - All tokens maintain context of sentence start")
print("   - Different heads specialize in different patterns")

print("\n4. IMPLICATIONS FOR YOUR PROJECT:")
print("    Model learns hierarchical features")
print("    Attention provides interpretability")
print("    Can identify which tokens influence predictions")
print("    Validates transformer architecture effectiveness")


print("\n ATTENTION VISUALIZATION COMPLETE!")
print("="*70)